In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Circuit Analysis Code Evaluation

This notebook evaluates the code implementing circuit analysis in the repository at `/net/scratch2/smallyan/leela_eval`.

## Setup and Initial Exploration

In [2]:
# Load environment variables from ~/.bashrc
import subprocess
result = subprocess.run(['bash', '-c', 'source ~/.bashrc && env'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if '=' in line and not line.startswith('_'):
        key, _, value = line.partition('=')
        os.environ[key] = value

# Verify HF_HOME
print(f"HF_HOME: {os.environ.get('HF_HOME', 'Not set')}")

HF_HOME: /net/projects2/chai-lab/shared_models


In [3]:
# Check CUDA availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA device count: {torch.cuda.device_count()}")

CUDA available: True
CUDA device: NVIDIA H100 NVL
CUDA device count: 1


In [4]:
# List the repository structure
repo_path = "/net/scratch2/smallyan/leela_eval"
for root, dirs, files in os.walk(repo_path):
    # Skip hidden directories and common non-essential directories
    dirs[:] = [d for d in dirs if not d.startswith('.') and d not in ['__pycache__', 'node_modules', '.git']]
    level = root.replace(repo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

leela_eval/
  lc0.onnx
  plan.md
  documentation.pdf
  .gitmodules
  pyproject.toml
  lc0-original.onnx
  768x15x24h-t82-swa-7464000.pb
  .gitignore
  CodeWalkthrough.md
  768x15x24h-t82-swa-7464000.pb.gz
  iteration_model/
    interesting_puzzles.pkl
    lc0.onnx
    lc0-random.onnx
    LD2.onnx
    unfiltered_puzzles.pkl
    lc0-original.onnx
  lc0_bin/
    lc0.tar.gz
  src/
    leela_logit_lens/
      __init__.py
      tournament/
        logit_lens_engine.py
        constants.py
      tools/
        evaluate_puzzles.py
        plotting_helpers.py
        utils.py
        sample_positions.py
        evaluate_concepts.py
        puzzle_history_augmentation.py
        concept_spec.py
        tournament.py
      core/
        leela_types.py
        leela_logit_lens.py
    leela_logit_lens.egg-info/
      requires.txt
      top_level.txt
      SOURCES.txt
      PKG-INFO
      dependency_links.txt
    searchless_chess/
      __init__.py
      utils.py
      engines/
        stockfish_eng

## Project Overview

Based on the Plan and CodeWalkthrough files:

**Objective**: Investigate how neural networks progressively build understanding across layers by extending the logit lens technique to analyze Leela Chess Zero's policy network.

**Core Notebooks to Evaluate** (as per CodeWalkthrough):
1. `notebooks/demo.ipynb` - Core functionality demo
2. `notebooks/figure1.ipynb` - Main figure plotting
3. `notebooks/puzzle_results.ipynb` - Puzzle solving results
4. `notebooks/tournament_results.ipynb` - Tournament Elo calculation  
5. `notebooks/policy_metrics.ipynb` - Policy distribution metrics
6. `notebooks/forgotten_puzzle_figure.ipynb` - Solution discovery/forgetting analysis

**Core Modules**:
- `src/leela_logit_lens/` - Core implementation
- `scripts/` - Evaluation scripts

In [5]:
# Install the package
os.chdir('/net/scratch2/smallyan/leela_eval')
import subprocess
result = subprocess.run(['pip', 'install', '-e', '.'], capture_output=True, text=True)
print("STDOUT:", result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
print("STDERR:", result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
print("Return code:", result.returncode)

STDOUT: me/smallyan/.local/lib/python3.12/site-packages (from scikit-learn->leela-interp@ git+https://github.com/HumanCompatibleAI/leela-interp.git->leela-logit-lens==0.0.1) (1.5.3)
  Building editable for leela-logit-lens (pyproject.toml): started
  Building editable for leela-logit-lens (pyproject.toml): finished with status 'done'
  Created wheel for leela-logit-lens: filename=leela_logit_lens-0.0.1-0.editable-py3-none-any.whl size=1512 sha256=62323122db518453238190ab3a462042ce360fd995df3092c311f9878b515e52
  Stored in directory: /tmp/pip-ephem-wheel-cache-xm0d6exe/wheels/dc/83/dc/13894fceb130a7d44870365657a137739ea22d16b66e5f5dc0
Successfully built leela-logit-lens
  Attempting uninstall: leela-logit-lens
    Found existing installation: leela-logit-lens 0.0.1
    Uninstalling leela-logit-lens-0.0.1:
      Successfully uninstalled leela-logit-lens-0.0.1

STDERR:   Running command git clone --filter=blob:none --quiet https://github.com/HumanCompatibleAI/leela-interp.git /tmp/pip-ins

In [6]:
# Verify installation and check data files
import os

# Check for required files
required_files = {
    'lc0-original.onnx': 'Model file',
    'data/puzzles.csv': 'Puzzles dataset',
    'data/eco_openings.pgn': 'ECO openings',
}

optional_files = {
    'data/cclr': 'CCRL dataset directory',
    'stockfish-8-linux/src/stockfish': 'Stockfish binary',
    'results/puzzle_results.csv': 'Pre-computed puzzle results',
    'results/concept_deltas.pkl': 'Pre-computed concept deltas',
    'results/tournament_games.pgn': 'Pre-computed tournament games'
}

print("=== Required Files ===")
for file, desc in required_files.items():
    exists = os.path.exists(file)
    print(f"{'✓' if exists else '✗'} {desc}: {file}")

print("\n=== Optional Files ===")
for file, desc in optional_files.items():
    exists = os.path.exists(file)
    print(f"{'✓' if exists else '✗'} {desc}: {file}")

=== Required Files ===
✓ Model file: lc0-original.onnx
✗ Puzzles dataset: data/puzzles.csv
✓ ECO openings: data/eco_openings.pgn

=== Optional Files ===
✓ CCRL dataset directory: data/cclr
✗ Stockfish binary: stockfish-8-linux/src/stockfish
✗ Pre-computed puzzle results: results/puzzle_results.csv
✗ Pre-computed concept deltas: results/concept_deltas.pkl
✗ Pre-computed tournament games: results/tournament_games.pgn


In [7]:
# Download puzzles.csv
import subprocess
os.makedirs('data', exist_ok=True)
result = subprocess.run(['wget', '-q', 'https://storage.googleapis.com/searchless_chess/data/puzzles.csv', '-P', 'data/'], 
                       capture_output=True, text=True)
print("Download puzzles.csv:", "Success" if result.returncode == 0 else f"Failed: {result.stderr}")
print("Puzzles file exists:", os.path.exists('data/puzzles.csv'))

Download puzzles.csv: Success
Puzzles file exists: True


## Notebook 1: demo.ipynb Evaluation

This notebook demonstrates the core functionality of the Leela Logit Lens.

In [8]:
# Evaluation tracking variables
evaluation_results = []

# Cell 1: Import leela_interp
print("=== Cell 1: Import leela_interp ===")
try:
    from leela_interp import Lc0sight, LeelaBoard
    print("✓ Cell executed successfully")
    evaluation_results.append({
        'notebook': 'demo.ipynb',
        'cell': 1,
        'description': 'Import leela_interp',
        'runnable': 'Y',
        'correct': 'Y',
        'redundant': 'N',
        'irrelevant': 'N',
        'error_note': ''
    })
except Exception as e:
    print(f"✗ Error: {e}")
    evaluation_results.append({
        'notebook': 'demo.ipynb',
        'cell': 1,
        'description': 'Import leela_interp',
        'runnable': 'N',
        'correct': 'N',
        'redundant': 'N',
        'irrelevant': 'N',
        'error_note': str(e)
    })

=== Cell 1: Import leela_interp ===


/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


✓ Cell executed successfully


In [9]:
# Cell 2: Set device to GPU
print("=== Cell 2: Set device ===")
try:
    # NOTE: Modified from original "cpu" to "cuda" per instructions
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device set to: {device}")
    print("✓ Cell executed successfully (modified to use GPU)")
    evaluation_results.append({
        'notebook': 'demo.ipynb',
        'cell': 2,
        'description': 'Set device',
        'runnable': 'Y',
        'correct': 'Y',
        'redundant': 'N',
        'irrelevant': 'N',
        'error_note': 'Modified to use GPU instead of CPU'
    })
except Exception as e:
    print(f"✗ Error: {e}")
    evaluation_results.append({
        'notebook': 'demo.ipynb',
        'cell': 2,
        'description': 'Set device',
        'runnable': 'N',
        'correct': 'N',
        'redundant': 'N',
        'irrelevant': 'N',
        'error_note': str(e)
    })

=== Cell 2: Set device ===
Device set to: cuda
✓ Cell executed successfully (modified to use GPU)


In [10]:
# Cell 3: Load model
print("=== Cell 3: Load model ===")
try:
    model = Lc0sight("lc0-original.onnx", device=device)
    print(f"Model loaded on device: {device}")
    print("✓ Cell executed successfully")
    evaluation_results.append({
        'notebook': 'demo.ipynb',
        'cell': 3,
        'description': 'Load Lc0sight model',
        'runnable': 'Y',
        'correct': 'Y',
        'redundant': 'N',
        'irrelevant': 'N',
        'error_note': ''
    })
except Exception as e:
    print(f"✗ Error: {e}")
    evaluation_results.append({
        'notebook': 'demo.ipynb',
        'cell': 3,
        'description': 'Load Lc0sight model',
        'runnable': 'N',
        'correct': 'N',
        'redundant': 'N',
        'irrelevant': 'N',
        'error_note': str(e)
    })

=== Cell 3: Load model ===
Using device: cuda


✗ Error: Lc0Model(
  (_lc0_model): GraphModule(
    (attn_body/transpose): OnnxTranspose()
    (initializers): Module()
    (attn_body/reshape): OnnxReshape()
    (attn_body/shape): OnnxShape()
    (attn_body/batch): OnnxSlice()
    (attn_body/pos_encoding_shape): OnnxConcat()
    (attn_body/expand): OnnxExpand()
    (attn_body/padded_input): OnnxConcat()
    (attn_body/reshape2): OnnxReshape()
    (attn_body/matmul): OnnxMatMul()
    (attn_body/add): OnnxBinaryMathOperation()
    (attn_body/mish/softplus): Softplus(beta=1.0, threshold=20.0)
    (attn_body/mish/tanh): OnnxFunction()
    (attn_body/mish): OnnxBinaryMathOperation()
    (attn_body/ma_gating/rehape1): OnnxReshape()
    (ip_mul_gate): OnnxBinaryMathOperation()
    (ip_add_gate): OnnxBinaryMathOperation()
    (attn_body/ma_gating/rehape2): OnnxReshape()
    (encoder0/mha/Q/w): OnnxMatMul()
    (encoder0/mha/Q/b): OnnxBinaryMathOperation()
    (encoder0/mha/Q/reshape): OnnxReshape()
    (encoder0/mha/Q/transpose): OnnxTranspo

In [11]:
# Cell 3 (retry): Load model on CPU as per original notebook
print("=== Cell 3 (retry): Load model on CPU ===")
try:
    device = "cpu"  # Original notebook uses CPU
    model = Lc0sight("lc0-original.onnx", device=device)
    print(f"Model loaded on device: {device}")
    print("✓ Cell executed successfully")
    # Update the evaluation result
    evaluation_results[-1] = {
        'notebook': 'demo.ipynb',
        'cell': 3,
        'description': 'Load Lc0sight model',
        'runnable': 'Y',
        'correct': 'Y',
        'redundant': 'N',
        'irrelevant': 'N',
        'error_note': 'Note: Original notebook uses CPU; GPU loading failed due to leela-interp library issue'
    }
except Exception as e:
    print(f"✗ Error: {e}")
    evaluation_results[-1] = {
        'notebook': 'demo.ipynb',
        'cell': 3,
        'description': 'Load Lc0sight model',
        'runnable': 'N',
        'correct': 'N',
        'redundant': 'N',
        'irrelevant': 'N',
        'error_note': str(e)
    }

=== Cell 3 (retry): Load model on CPU ===
Using device: cpu


✗ Error: Lc0Model(
  (_lc0_model): GraphModule(
    (attn_body/transpose): OnnxTranspose()
    (initializers): Module()
    (attn_body/reshape): OnnxReshape()
    (attn_body/shape): OnnxShape()
    (attn_body/batch): OnnxSlice()
    (attn_body/pos_encoding_shape): OnnxConcat()
    (attn_body/expand): OnnxExpand()
    (attn_body/padded_input): OnnxConcat()
    (attn_body/reshape2): OnnxReshape()
    (attn_body/matmul): OnnxMatMul()
    (attn_body/add): OnnxBinaryMathOperation()
    (attn_body/mish/softplus): Softplus(beta=1.0, threshold=20.0)
    (attn_body/mish/tanh): OnnxFunction()
    (attn_body/mish): OnnxBinaryMathOperation()
    (attn_body/ma_gating/rehape1): OnnxReshape()
    (ip_mul_gate): OnnxBinaryMathOperation()
    (ip_add_gate): OnnxBinaryMathOperation()
    (attn_body/ma_gating/rehape2): OnnxReshape()
    (encoder0/mha/Q/w): OnnxMatMul()
    (encoder0/mha/Q/b): OnnxBinaryMathOperation()
    (encoder0/mha/Q/reshape): OnnxReshape()
    (encoder0/mha/Q/transpose): OnnxTranspo

In [12]:
# Check nnsight version
import subprocess
result = subprocess.run(['pip', 'show', 'nnsight'], capture_output=True, text=True)
print(result.stdout)

Name: nnsight
Version: 0.2.21
Summary: Package for interpreting and manipulating the internals of deep learning models.
Home-page: https://github.com/ndif-team/nnsight
Author: 
Author-email: Jaden Fiotto-Kaufman <jadenfk@outlook.com>
License: 
Location: /home/smallyan/.local/lib/python3.12/site-packages
Requires: accelerate, diffusers, einops, protobuf, pydantic, python-socketio, sentencepiece, tokenizers, torch, torchvision, transformers
Required-by: leela-interp



In [13]:
# Try to install an older version of nnsight that's compatible
import subprocess
result = subprocess.run(['pip', 'install', 'nnsight==0.1.3', '-q'], capture_output=True, text=True)
print("STDOUT:", result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
print("STDERR:", result.stderr[-1000:] if len(result.stderr) > 1000 else result.stderr)
print("Return code:", result.returncode)

STDOUT: 
STDERR: ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
leela-interp 0.0.1 requires nnsight==0.2.*, but you have nnsight 0.1.3 which is incompatible.

Return code: 0


In [14]:
# Try reinstalling the leela-interp with its nnsight dependency
import subprocess
result = subprocess.run(['pip', 'install', 'nnsight==0.2.16', '-q'], capture_output=True, text=True)
print("Return code:", result.returncode)
print("STDERR:", result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)

Return code: 0
STDERR: 


In [15]:
# Need to restart the kernel to pick up new nnsight version
import importlib
import sys

# Remove cached modules to try loading fresh version
mods_to_remove = [k for k in sys.modules.keys() if 'nnsight' in k or 'leela' in k]
for mod in mods_to_remove:
    del sys.modules[mod]

# Try importing again
try:
    from leela_interp import Lc0sight, LeelaBoard
    from leela_logit_lens import LeelaLogitLens
    
    device = "cpu"
    model = Lc0sight("lc0-original.onnx", device=device)
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error: {type(e).__name__}: {str(e)[:500]}")

Using device: cpu


Error: AttributeError: Lc0Model(
  (_lc0_model): GraphModule(
    (attn_body/transpose): OnnxTranspose()
    (initializers): Module()
    (attn_body/reshape): OnnxReshape()
    (attn_body/shape): OnnxShape()
    (attn_body/batch): OnnxSlice()
    (attn_body/pos_encoding_shape): OnnxConcat()
    (attn_body/expand): OnnxExpand()
    (attn_body/padded_input): OnnxConcat()
    (attn_body/reshape2): OnnxReshape()
    (attn_body/matmul): OnnxMatMul()
    (attn_body/add): OnnxBinaryMathOperation()
    (attn_body/mish/softplu


In [16]:
# Let me check what leela-interp expects by looking at the library source
import subprocess
result = subprocess.run(['pip', 'show', '-f', 'leela-interp'], capture_output=True, text=True)
# Find location
import pkg_resources
try:
    loc = pkg_resources.get_distribution("leela-interp").location
    print(f"Location: {loc}")
except:
    print("leela-interp location not found")

# List the location of the lib
result = subprocess.run(['find', '/home/smallyan/.local/lib/python3.12/site-packages/leela_interp', '-name', '*.py'], capture_output=True, text=True)
print(result.stdout)

/tmp/ipykernel_3526501/397038253.py:5: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


Location: /home/smallyan/.conda/envs/meta/lib/python3.11/site-packages
/home/smallyan/.local/lib/python3.12/site-packages/leela_interp/tools/patching.py
/home/smallyan/.local/lib/python3.12/site-packages/leela_interp/tools/piece_movement_heads.py
/home/smallyan/.local/lib/python3.12/site-packages/leela_interp/tools/figure_helpers.py
/home/smallyan/.local/lib/python3.12/site-packages/leela_interp/tools/activations.py
/home/smallyan/.local/lib/python3.12/site-packages/leela_interp/tools/play.py
/home/smallyan/.local/lib/python3.12/site-packages/leela_interp/tools/probing.py
/home/smallyan/.local/lib/python3.12/site-packages/leela_interp/tools/attention.py
/home/smallyan/.local/lib/python3.12/site-packages/leela_interp/core/utils.py
/home/smallyan/.local/lib/python3.12/site-packages/leela_interp/core/uci_to_idx.py
/home/smallyan/.local/lib/python3.12/site-packages/leela_interp/core/lc0.py
/home/smallyan/.local/lib/python3.12/site-packages/leela_interp/core/iceberg_board.py
/home/smallyan/

In [17]:
# Try installing an older compatible nnsight version
import subprocess

# The leela-interp library requires nnsight==0.2.* but the _envoy attribute was removed in newer versions
# Let's try versions from 0.2.0 to 0.2.10
for version in ['0.2.10', '0.2.8', '0.2.5', '0.2.0']:
    result = subprocess.run(['pip', 'install', f'nnsight=={version}', '-q'], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"Installed nnsight=={version}")
        break
    else:
        print(f"Failed to install {version}: {result.stderr[-200:]}")

Installed nnsight==0.2.10


In [18]:
# Need to restart Python session to pick up new version
# Let's check if we can simply use the nnsight version in the error
import subprocess
result = subprocess.run(['pip', 'show', 'nnsight'], capture_output=True, text=True)
print(result.stdout[:300])

# Try reloading modules
import sys
mods_to_remove = [k for k in sys.modules.keys() if 'nnsight' in k or 'leela' in k]
for mod in mods_to_remove:
    del sys.modules[mod]

try:
    import nnsight
    print(f"nnsight version after reload: {nnsight.__version__}")
except:
    pass

Name: nnsight
Version: 0.2.10
Summary: Package for interpreting and manipulating the internals of deep learning models.
Home-page: https://github.com/JadenFiotto-Kaufman/nnsight
Author: 
Author-email: Jaden Fiotto-Kaufman <jadenfk@outlook.com>
License: 
Location: /home/smallyan/.local/lib/python3.12
nnsight version after reload: 0.5.2
